# Notebook 04 — Experiments and Selection (Semana 4)

## Objetivo
Executar um conjunto pequeno, rápido e rastreável de experimentos comparáveis para orientar o Notebook 05.  
Este notebook **não** busca SOTA; ele existe para **sanity check + seleção preliminar/candidata**, dependendo do ambiente disponível.

## Escopo
- Comparar abordagens de classificação com regras claras de seleção.
- Salvar tabela de resultados, configs completas e decisão formal.
- Manter a etapa reproduzível e auditável.
- **Não** misturar duas trilhas finais no mesmo notebook.

## Entradas
- `data/processed/train.csv`
- `data/processed/val.csv`
- `data/processed/test.csv`
- `data/processed/label_map.json`
- Preferir `data/processed/target_config_effective.json`
- Fallback: `data/processed/target_config.json`

## Saídas obrigatórias
- `reports/experiments_table.csv`
- `reports/experiments_configs.json`
- `reports/selection_decision.md`
- `reports/plots/cm_*.png` (quando aplicável)

## Regras operacionais
- Tudo relativo ao **repo root** do `pimple`
- Seed fixa + configs salvas
- Um único fluxo oficial de execução
- Se `torchvision` estiver ausente, o notebook opera em modo **`sklearn_only_preliminary`**
- Segmentação fica fora deste notebook

In [1]:
# CÉLULA 01 — Imports + detecção de torch/torchvision (sem pegadinha)
import os
import json
import time
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

TORCH_AVAILABLE = False
TORCHVISION_AVAILABLE = False
TORCH_IMPORT_ERROR = None
TORCHVISION_IMPORT_ERROR = None

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    TORCH_AVAILABLE = True
except Exception as e:
    TORCH_IMPORT_ERROR = repr(e)

try:
    from torchvision import transforms
    from torchvision.models import resnet18, ResNet18_Weights
    TORCHVISION_AVAILABLE = True
except Exception as e:
    TORCHVISION_IMPORT_ERROR = repr(e)

print("TORCH_AVAILABLE:", TORCH_AVAILABLE)
if not TORCH_AVAILABLE:
    print("Torch import error:", TORCH_IMPORT_ERROR)

print("TORCHVISION_AVAILABLE:", TORCHVISION_AVAILABLE)
if not TORCHVISION_AVAILABLE:
    print("Torchvision import error:", TORCHVISION_IMPORT_ERROR)


TORCH_AVAILABLE: True
TORCHVISION_AVAILABLE: True


In [2]:
# CÉLULA 02 — Guard claro: este NB04 roda sklearn-only quando torchvision não existe
USE_TORCH = bool(TORCH_AVAILABLE)
USE_TORCHVISION = bool(TORCHVISION_AVAILABLE)

print("USE_TORCH:", USE_TORCH)
print("USE_TORCHVISION:", USE_TORCHVISION)

if not USE_TORCHVISION:
    print("[INFO] torchvision ausente → Notebook 04 rodando apenas experimentos sklearn (ok para Semana 4).")


USE_TORCH: True
USE_TORCHVISION: True


In [3]:
# CÉLULA 03 — Repo root robusto + paths do projeto

def _looks_like_repo_root(p: Path) -> bool:
    return (
        (p / "data" / "raw" / "lesions" / "images").exists()
        and (p / "data" / "processed").exists()
        and (p / "reports").exists()
    )

def _normalize_repo_candidate(p: Path) -> Optional[Path]:
    p = p.expanduser().resolve()
    if _looks_like_repo_root(p):
        return p
    if _looks_like_repo_root(p / "pimple"):
        return (p / "pimple").resolve()
    return None

def find_project_root_robust() -> Path:
    env_root = os.environ.get("PIMPLE_PROJECT_ROOT") or os.environ.get("PROJECT_ROOT")
    if env_root:
        normalized = _normalize_repo_candidate(Path(env_root))
        if normalized is not None:
            return normalized
        raise FileNotFoundError(
            f"[ERRO] Env PROJECT_ROOT/PIMPLE_PROJECT_ROOT aponta para {env_root}, "
            "mas não parece ser a raiz do repo pimple nem o diretório pai que contém a pasta pimple."
        )

    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        normalized = _normalize_repo_candidate(base)
        if normalized is not None:
            return normalized

    raise FileNotFoundError(
        "Não consegui localizar a raiz do repositório pimple.\n"
        "Dica: defina os.environ['PIMPLE_PROJECT_ROOT'] = r'CAMINHO_PARA_O_REPO_PIMPLE' e rode de novo."
    )

PROJECT_ROOT = find_project_root_robust()

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "lesions"
IMAGES_DIR = RAW_DIR / "images"
MASKS_DIR = RAW_DIR / "masks"
GT_CSV = RAW_DIR / "GroundTruth.csv"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
PLOTS_DIR = REPORTS_DIR / "plots"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMAGES_DIR:", IMAGES_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("REPORTS_DIR:", REPORTS_DIR)

assert IMAGES_DIR.exists(), f"IMAGES_DIR não existe: {IMAGES_DIR}"
assert PROCESSED_DIR.exists(), f"PROCESSED_DIR não existe: {PROCESSED_DIR}"

PROJECT_ROOT: C:\Users\win\Documents\GitHub\pimple
IMAGES_DIR: C:\Users\win\Documents\GitHub\pimple\data\raw\lesions\images
PROCESSED_DIR: C:\Users\win\Documents\GitHub\pimple\data\processed
REPORTS_DIR: C:\Users\win\Documents\GitHub\pimple\reports


In [4]:
# CÉLULA 04 — Leitura de contratos (preferir effective) + label_map robusto

def read_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def deep_get(d: Dict[str, Any], keys: List[str]) -> Optional[Any]:
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    common_parents = ["target", "target_definition", "target_config", "columns", "schema", "contract", "data"]
    for parent in common_parents:
        if parent in d and isinstance(d[parent], dict):
            for k in keys:
                if k in d[parent] and d[parent][k] is not None:
                    return d[parent][k]
    return None

def normalize_label_map_robust(lm: Any, label_cols: List[str]) -> Tuple[List[str], Dict[str, int], str]:
    if isinstance(lm, dict):
        for k in ["idx_to_label", "index_to_label", "id2label", "classes", "labels"]:
            v = lm.get(k, None)
            if isinstance(v, list) and all(isinstance(x, str) for x in v):
                idx_to_label = v
                return idx_to_label, {lab: i for i, lab in enumerate(idx_to_label)}, f"wrapped_list:{k}"
        for k in ["label_to_idx", "label2id", "label2idx"]:
            v = lm.get(k, None)
            if isinstance(v, dict) and all(isinstance(val, int) for val in v.values()):
                label_to_idx = dict(v)
                idx_to_label = [None] * (max(label_to_idx.values()) + 1)
                for lab, i in label_to_idx.items():
                    idx_to_label[i] = lab
                return idx_to_label, label_to_idx, f"wrapped_dict:{k}"

    if isinstance(lm, dict) and len(lm) > 0 and all(str(k).isdigit() for k in lm.keys()):
        idx_to_label = [lm[str(i)] for i in range(len(lm))]
        return idx_to_label, {lab: i for i, lab in enumerate(idx_to_label)}, "digit_key_dict"

    if isinstance(lm, dict) and len(lm) > 0 and all(isinstance(v, int) for v in lm.values()):
        label_to_idx = dict(lm)
        idx_to_label = [None] * (max(label_to_idx.values()) + 1)
        for lab, i in label_to_idx.items():
            idx_to_label[i] = lab
        return idx_to_label, label_to_idx, "label_to_idx_dict"

    if isinstance(lm, list) and all(isinstance(x, str) for x in lm):
        idx_to_label = lm
        return idx_to_label, {lab: i for i, lab in enumerate(idx_to_label)}, "list"

    if isinstance(lm, dict) and len(lm) > 0 and all(isinstance(v, str) for v in lm.values()):
        keys = set(lm.keys())
        if all(c in keys for c in label_cols):
            idx_to_label = [lm[c] for c in label_cols]
            return idx_to_label, {lab: i for i, lab in enumerate(idx_to_label)}, "col_to_label_aligned_by_label_cols"
        lower_map = {str(k).lower(): v for k, v in lm.items()}
        if all(str(c).lower() in lower_map for c in label_cols):
            idx_to_label = [lower_map[str(c).lower()] for c in label_cols]
            return idx_to_label, {lab: i for i, lab in enumerate(idx_to_label)}, "col_to_label_lower_aligned_by_label_cols"

    idx_to_label = list(label_cols)
    return idx_to_label, {lab: i for i, lab in enumerate(idx_to_label)}, "fallback_use_label_cols"

target_cfg_effective_path = PROCESSED_DIR / "target_config_effective.json"
target_cfg_path = PROCESSED_DIR / "target_config.json"
label_map_path = PROCESSED_DIR / "label_map.json"

assert label_map_path.exists(), f"label_map.json não encontrado: {label_map_path}"
label_map = read_json(label_map_path)

if target_cfg_effective_path.exists():
    target_cfg = read_json(target_cfg_effective_path)
    target_cfg_source = str(target_cfg_effective_path.relative_to(PROJECT_ROOT))
else:
    assert target_cfg_path.exists(), f"target_config.json não encontrado: {target_cfg_path}"
    target_cfg = read_json(target_cfg_path)
    target_cfg_source = str(target_cfg_path.relative_to(PROJECT_ROOT))

print("Target config source:", target_cfg_source)

mode = deep_get(target_cfg, ["mode", "task_mode"])
target_encoding = deep_get(target_cfg, ["target_encoding", "encoding", "label_encoding"])
label_cols = deep_get(target_cfg, ["label_cols", "labels", "label_columns", "target_cols"])
image_col = deep_get(target_cfg, ["image_col", "image_column", "image", "filename_col", "path_col"])

if isinstance(label_cols, tuple):
    label_cols = list(label_cols)

assert isinstance(label_cols, list) and len(label_cols) > 1, f"label_cols inválido: {label_cols}"
assert isinstance(image_col, str) and len(image_col) > 0, f"image_col inválido: {image_col}"

if mode is None:
    print("[WARN] mode não encontrado. Assumindo single_label (coerente com one-hot).")
else:
    assert mode == "single_label", f"Notebook 04 assume single_label. Encontrado: {mode}"

if target_encoding is None:
    print("[WARN] target_encoding não encontrado. Prosseguindo.")
else:
    if target_encoding != "one_hot_multiclass":
        print("[WARN] target_encoding diferente do esperado:", target_encoding)

IDX_TO_LABEL, LABEL_TO_IDX, LABEL_MAP_STRATEGY = normalize_label_map_robust(label_map, label_cols)

print("mode:", mode)
print("target_encoding:", target_encoding)
print("image_col:", image_col)
print("label_cols:", label_cols)
print("label_map strategy:", LABEL_MAP_STRATEGY)
print("labels:", IDX_TO_LABEL)

mask_policy = None
if isinstance(target_cfg, dict):
    mask_policy = target_cfg.get("mask_policy", None)
    if mask_policy is None and isinstance(target_cfg.get("mask"), dict):
        mask_policy = target_cfg["mask"].get("policy", None)
print("mask_policy:", mask_policy)


Target config source: data\processed\target_config_effective.json
[WARN] target_encoding não encontrado. Prosseguindo.
mode: single_label
target_encoding: None
image_col: image_stem
label_cols: ['mel', 'nv', 'bcc', 'akiec', 'bkl', 'df', 'vasc']
label_map strategy: wrapped_list:index_to_label
labels: ['mel', 'nv', 'bcc', 'akiec', 'bkl', 'df', 'vasc']
mask_policy: {'mask_coverage_ratio_in_clean': 1.0, 'mask_required_for_inclusion': False, 'notes': 'Máscaras são opcionais no dataset_clean; usadas apenas se a trilha de segmentação for escolhida.', 'resolution_strategy': {'fallback_by_stem': True, 'from_csv_if_available': False, 'suffixes_used': ['', '_mask', '-mask', '_seg', '-seg', '_segmentation', '-segmentation', '_lesion', '_lesion_mask', '_binary', '_annotation', '_ann']}, 'segmentation_considered': True}


In [5]:
# CÉLULA 05 — Carregar splits + one-hot -> class index y
train_path = PROCESSED_DIR / "train.csv"
val_path = PROCESSED_DIR / "val.csv"
test_path = PROCESSED_DIR / "test.csv"

assert train_path.exists() and val_path.exists() and test_path.exists(), "train/val/test.csv não encontrados em data/processed/"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert image_col in df.columns, f"[{name}] falta image_col={image_col}"
    missing = [c for c in label_cols if c not in df.columns]
    assert len(missing) == 0, f"[{name}] faltam label_cols: {missing}"

print("Shapes:", train_df.shape, val_df.shape, test_df.shape)

def one_hot_to_y(df: pd.DataFrame, cols: List[str]) -> np.ndarray:
    y = df[cols].values
    s = y.sum(axis=1)
    if not np.allclose(s, 1.0, atol=1e-6):
        bad = np.where(~np.isclose(s, 1.0, atol=1e-6))[0][:10]
        raise ValueError(f"one-hot inválido: soma != 1. Ex idx: {bad}")
    return np.argmax(y, axis=1).astype(int)

y_train = one_hot_to_y(train_df, label_cols)
y_val = one_hot_to_y(val_df, label_cols)
y_test = one_hot_to_y(test_df, label_cols)

print("y_train distribution:", np.bincount(y_train, minlength=len(label_cols)))


Shapes: (7011, 12) (1502, 12) (1502, 12)
y_train distribution: [ 779 4693  360  229  769   81  100]


In [6]:
# CÉLULA 06 — Seed fixa + device + Timer
GLOBAL_SEED = int(target_cfg.get("seed", 42))

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(GLOBAL_SEED)

DEVICE = "cpu"
if TORCH_AVAILABLE and torch.cuda.is_available():
    DEVICE = "cuda"
elif TORCH_AVAILABLE and getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"

print("GLOBAL_SEED:", GLOBAL_SEED)
print("DEVICE:", DEVICE)

@dataclass
class Timer:
    start: float = 0.0
    end: float = 0.0
    def __enter__(self):
        self.start = time.time()
        return self
    def __exit__(self, exc_type, exc, tb):
        self.end = time.time()
    @property
    def seconds(self) -> float:
        return float(self.end - self.start)


GLOBAL_SEED: 42
DEVICE: cuda


In [7]:
# CÉLULA 07 — Métricas + plot confusion matrix
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> Dict[str, Any]:
    acc = float(accuracy_score(y_true, y_pred))
    f1m = float(f1_score(y_true, y_pred, average="macro"))
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    return {"accuracy": acc, "f1_macro": f1m, "confusion_matrix": cm}

def plot_confusion_matrix(cm: np.ndarray, labels: List[str], title: str, save_path: Path) -> None:
    fig = plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(labels))
    plt.xticks(tick_marks, labels, rotation=45, ha="right")
    plt.yticks(tick_marks, labels)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    fig.savefig(save_path, dpi=160, bbox_inches="tight")
    plt.close(fig)


In [8]:
# CÉLULA 08 — resolve_image_path robusto (padrão NB03)
ALLOWED_IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"]

def resolve_image_path(img_value: Any) -> Path:
    if pd.isna(img_value):
        raise ValueError("image_col contém NaN.")

    s = str(img_value).strip().replace("\\", "/")
    p = Path(s)

    # 0) se já existe (absoluto/relativo ao cwd), usa direto
    try:
        p2 = p.expanduser()
        if p2.exists():
            return p2.resolve()
    except Exception:
        pass

    # 1) basename
    fname = p.name
    c1 = IMAGES_DIR / fname
    if c1.exists():
        return c1

    # 2) relativo dentro de images/
    c2 = IMAGES_DIR / s
    if c2.exists():
        return c2

    # 3) stem
    stem = Path(fname).stem
    for ext in ALLOWED_IMAGE_EXTS:
        c3 = IMAGES_DIR / f"{stem}{ext}"
        if c3.exists():
            return c3

    # 4) glob fallback
    matches = sorted(IMAGES_DIR.glob(f"{stem}.*"))
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Imagem não encontrada para valor={img_value}. Tentativas: {c1}, {c2}, stem={stem}")

# sanity
ok = 0
for v in train_df[image_col].head(20).tolist():
    try:
        _ = resolve_image_path(v)
        ok += 1
    except Exception as e:
        print("FAIL sample:", v, "=>", repr(e))
print(f"[SANITY] resolve_image_path: {ok}/20 resolvidas.")


[SANITY] resolve_image_path: 20/20 resolvidas.


In [9]:
# CÉLULA 09 — Features para sklearn (flatten)
def load_image_as_feature(img_path: Path, size: int) -> np.ndarray:
    img = Image.open(img_path).convert("RGB").resize((size, size))
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr.reshape(-1)

def make_features(df: pd.DataFrame, size: int) -> np.ndarray:
    X = np.zeros((len(df), size * size * 3), dtype=np.float32)
    for i, v in enumerate(df[image_col].values):
        p = resolve_image_path(v)
        X[i] = load_image_as_feature(p, size=size)
    return X


In [10]:
# CÉLULA 10 — fit_eval_sklearn (LogReg + LinearSVC; uma única interface)
from sklearn.svm import LinearSVC

def fit_eval_sklearn(train_df, y_train, val_df, y_val, cfg: Dict[str, Any]) -> Dict[str, Any]:
    family = str(cfg["family"])
    size = int(cfg["input_size"])
    class_weight = cfg.get("class_weight", None)

    Xtr = make_features(train_df, size=size)
    Xva = make_features(val_df, size=size)

    scaler = StandardScaler(with_mean=True, with_std=True)

    if family == "sklearn_logreg_flatten":
        base_params = LogisticRegression().get_params()
        clf_kwargs = {
            "max_iter": int(cfg.get("max_iter", 250)),
            "solver": "lbfgs",
            "class_weight": class_weight,
            "random_state": int(cfg.get("seed", GLOBAL_SEED)),
        }
        clf_kwargs = {k: v for k, v in clf_kwargs.items() if k in base_params}
        clf = LogisticRegression(**clf_kwargs)

    elif family == "sklearn_linearsvc_flatten":
        base_params = LinearSVC().get_params()
        clf_kwargs = {
            "C": float(cfg.get("C", 1.0)),
            "class_weight": class_weight,
            "max_iter": int(cfg.get("max_iter", 3000)),
            "random_state": int(cfg.get("seed", GLOBAL_SEED)),
        }
        clf_kwargs = {k: v for k, v in clf_kwargs.items() if k in base_params}
        clf = LinearSVC(**clf_kwargs)

    else:
        raise ValueError(f"Família sklearn não suportada: {family}")

    pipe = Pipeline([
        ("scaler", scaler),
        ("clf", clf),
    ])

    pipe.fit(Xtr, y_train)
    y_pred = pipe.predict(Xva)

    m = compute_metrics(y_val, y_pred, n_classes=len(label_cols))
    return {
        "model_obj": pipe,
        "val_metrics": {"accuracy": m["accuracy"], "f1_macro": m["f1_macro"]},
        "val_confusion_matrix": m["confusion_matrix"],
    }

In [11]:
# CÉLULA 11 — Registry único de experimentos (sem override no final)
EXPERIMENTS: Dict[str, Dict[str, Any]] = {
    "sk_logreg_32": {
        "family": "sklearn_logreg_flatten",
        "input_size": 32,
        "class_weight": None,
        "max_iter": 250,
        "seed": GLOBAL_SEED,
        "selection_scope": "sanity_reference",
        "notes": "Baseline rápido e simples.",
    },
    "sk_logreg_32_bal": {
        "family": "sklearn_logreg_flatten",
        "input_size": 32,
        "class_weight": "balanced",
        "max_iter": 250,
        "seed": GLOBAL_SEED,
        "selection_scope": "sanity_reference",
        "notes": "Mesmo baseline com class_weight=balanced.",
    },
    "sk_logreg_64": {
        "family": "sklearn_logreg_flatten",
        "input_size": 64,
        "class_weight": None,
        "max_iter": 250,
        "seed": GLOBAL_SEED,
        "selection_scope": "sanity_reference",
        "notes": "Aumenta resolução mantendo pipeline linear.",
    },
    "sk_logreg_64_bal": {
        "family": "sklearn_logreg_flatten",
        "input_size": 64,
        "class_weight": "balanced",
        "max_iter": 250,
        "seed": GLOBAL_SEED,
        "selection_scope": "sanity_reference",
        "notes": "Teste de resolução maior + balanced.",
    },
    "svm_64_bal": {
        "family": "sklearn_linearsvc_flatten",
        "input_size": 64,
        "class_weight": "balanced",
        "C": 1.0,
        "max_iter": 3000,
        "seed": GLOBAL_SEED,
        "selection_scope": "sanity_reference",
        "notes": "Família linear alternativa para comparação rápida.",
    },
}

print("Experiments:", list(EXPERIMENTS.keys()))

Experiments: ['sk_logreg_32', 'sk_logreg_32_bal', 'sk_logreg_64', 'sk_logreg_64_bal', 'svm_64_bal']


In [12]:
# CÉLULA 12 — Runner único (somente sklearn) + helpers de seleção/execução

def infer_selection_mode_from_registry(experiments: Dict[str, Dict[str, Any]]) -> str:
    """
    Decide o modo do NB04 com base no que REALMENTE foi registrado no notebook,
    e não apenas no ambiente disponível.
    """
    has_non_sklearn = any(
        not str(cfg.get("family", "")).startswith("sklearn_")
        for cfg in experiments.values()
    )
    has_candidate_scope = any(
        str(cfg.get("selection_scope", "")) == "candidate_for_nb05"
        for cfg in experiments.values()
    )

    if has_non_sklearn or has_candidate_scope:
        return "candidate_selection"
    return "sklearn_only_preliminary"


def infer_execution_device(family: str) -> str:
    """
    Define o backend real de execução por família.
    sklearn roda em CPU; trilhas torch podem usar DEVICE.
    """
    family = str(family)
    if family.startswith("sklearn_"):
        return "cpu"
    return DEVICE


def run_one_experiment(exp_id: str, cfg: Dict[str, Any]) -> Dict[str, Any]:
    cfg_full = dict(cfg)
    cfg_full["exp_id"] = exp_id
    cfg_full["target_config_source"] = target_cfg_source
    cfg_full["project_root"] = str(PROJECT_ROOT)
    cfg_full["labels"] = IDX_TO_LABEL
    cfg_full["mask_policy"] = mask_policy
    cfg_full["selection_scope"] = cfg_full.get("selection_scope", "sanity_reference")

    family = str(cfg_full.get("family"))
    execution_device = infer_execution_device(family)
    cfg_full["execution_device"] = execution_device

    with Timer() as t:
        if family in {"sklearn_logreg_flatten", "sklearn_linearsvc_flatten"}:
            out = fit_eval_sklearn(train_df, y_train, val_df, y_val, cfg_full)
        else:
            raise ValueError(f"Família desconhecida para o NB04 atual: {family}")

    rec = {
        "experiment_id": exp_id,
        "family": family,
        "selection_scope": cfg_full.get("selection_scope"),
        "input_size": int(cfg_full.get("input_size")),
        "normalization": cfg_full.get("normalization", None),
        "class_weight": cfg_full.get("class_weight", None),
        "seed": int(cfg_full.get("seed", GLOBAL_SEED)),
        "device": execution_device,
        "val_accuracy": float(out["val_metrics"]["accuracy"]),
        "val_f1_macro": float(out["val_metrics"]["f1_macro"]),
        "runtime_sec": float(t.seconds),
        "val_confusion_matrix_path": None,
        "error": None,
    }

    cm = out.get("val_confusion_matrix")
    if cm is not None:
        cm_path = PLOTS_DIR / f"cm_{exp_id}.png"
        plot_confusion_matrix(
            cm,
            IDX_TO_LABEL,
            title=f"Confusion Matrix (val) — {exp_id}",
            save_path=cm_path,
        )
        rec["val_confusion_matrix_path"] = str(cm_path.relative_to(PROJECT_ROOT))

    return {"record": rec, "config": cfg_full}

In [ ]:
# CÉLULA 13 — Executar todos os experimentos (uma única execução) + ordenar por F1 macro

results: List[Dict[str, Any]] = []
configs_out: Dict[str, Any] = {}

for exp_id, cfg in EXPERIMENTS.items():
    print("\n==============================")
    print("RUN:", exp_id)
    print("==============================")
    try:
        r = run_one_experiment(exp_id, cfg)
        results.append(r["record"])
        configs_out[exp_id] = r["config"]
        print(
            "OK:",
            exp_id,
            "| val_f1_macro =",
            round(r["record"]["val_f1_macro"], 6),
            "| device =",
            r["record"]["device"],
        )
    except Exception as e:
        err = repr(e)
        results.append({
            "experiment_id": exp_id,
            "family": cfg.get("family"),
            "selection_scope": cfg.get("selection_scope", "sanity_reference"),
            "input_size": cfg.get("input_size"),
            "normalization": cfg.get("normalization", None),
            "class_weight": cfg.get("class_weight", None),
            "seed": int(cfg.get("seed", GLOBAL_SEED)),
            "device": infer_execution_device(cfg.get("family", "")),
            "val_accuracy": None,
            "val_f1_macro": None,
            "runtime_sec": None,
            "val_confusion_matrix_path": None,
            "error": err,
        })
        configs_out[exp_id] = dict(
            cfg,
            exp_id=exp_id,
            error=err,
            execution_device=infer_execution_device(cfg.get("family", "")),
        )
        print("FAILED:", exp_id, "|", err)

experiments_df = pd.DataFrame(results)

if experiments_df.empty:
    raise RuntimeError("Nenhum resultado foi produzido no NB04.")

experiments_df = experiments_df.sort_values(
    by=["val_f1_macro", "val_accuracy", "runtime_sec"],
    ascending=[False, False, True],
    na_position="last",
).reset_index(drop=True)

experiments_df["rank_f1"] = experiments_df["val_f1_macro"].rank(
    ascending=False,
    method="min",
)

# Ajuste cosmético para exibição
experiments_df["class_weight"] = experiments_df["class_weight"].astype(object)
experiments_df.loc[experiments_df["class_weight"].isna(), "class_weight"] = None

selection_mode = infer_selection_mode_from_registry(EXPERIMENTS)
print("selection_mode inferido:", selection_mode)

display_cols = [
    "rank_f1",
    "experiment_id",
    "family",
    "selection_scope",
    "input_size",
    "class_weight",
    "device",
    "val_f1_macro",
    "val_accuracy",
    "runtime_sec",
    "error",
]
experiments_df[display_cols]


RUN: sk_logreg_32


c:\Users\win\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 250 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=250).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


OK: sk_logreg_32 | val_f1_macro = 0.3816 | device = cpu

RUN: sk_logreg_32_bal


c:\Users\win\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 250 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=250).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


OK: sk_logreg_32_bal | val_f1_macro = 0.36343 | device = cpu

RUN: sk_logreg_64


...

OK: sk_logreg_64 | val_f1_macro = 0.336021 | device = cpu

RUN: sk_logreg_64_bal


c:\Users\win\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 250 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=250).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


OK: sk_logreg_64_bal | val_f1_macro = 0.350698 | device = cpu

RUN: svm_64_bal
OK: svm_64_bal | val_f1_macro = 0.286749 | device = cpu
selection_mode inferido: sklearn_only_preliminary


c:\Users\win\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,rank_f1,experiment_id,family,selection_scope,input_size,class_weight,device,val_f1_macro,val_accuracy,runtime_sec,error
0,1.0,sk_logreg_32,sklearn_logreg_flatten,sanity_reference,32,NaN,cpu,0.381600,0.657124,30.036514,None
1,2.0,sk_logreg_32_bal,sklearn_logreg_flatten,sanity_reference,32,balanced,cpu,0.363430,0.577230,29.408505,None
2,3.0,sk_logreg_64_bal,sklearn_logreg_flatten,sanity_reference,64,balanced,cpu,0.350698,0.617177,38.356752,None
3,4.0,sk_logreg_64,sklearn_logreg_flatten,sanity_reference,64,NaN,cpu,0.336021,0.633822,38.372390,None
4,5.0,svm_64_bal,sklearn_linearsvc_flatten,sanity_reference,64,balanced,cpu,0.286749,0.614514,571.009383,None


In [14]:
# CÉLULA 14 — Salvar entregáveis obrigatórios: experiments_table.csv + experiments_configs.json

table_path = REPORTS_DIR / "experiments_table.csv"
configs_path = REPORTS_DIR / "experiments_configs.json"

experiments_df.to_csv(table_path, index=False)

selection_mode = infer_selection_mode_from_registry(EXPERIMENTS)

payload = {
    "meta": {
        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "project_root": str(PROJECT_ROOT),
        "target_config_source": target_cfg_source,
        "label_cols": label_cols,
        "image_col": image_col,
        "labels": IDX_TO_LABEL,
        "seed": GLOBAL_SEED,
        "default_device_available": DEVICE,
        "torch_available": TORCH_AVAILABLE,
        "torchvision_available": TORCHVISION_AVAILABLE,
        "mask_policy": mask_policy,
        "selection_mode": selection_mode,
        "note": (
            "Registry atual contém apenas experimentos sklearn/sanity. "
            "Resultados devem ser interpretados como seleção preliminar/baseline."
            if selection_mode == "sklearn_only_preliminary"
            else "Registry inclui candidatos reais além de sanity baselines."
        ),
    },
    "experiments": configs_out,
}

with open(configs_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

print("Saved:", table_path.relative_to(PROJECT_ROOT))
print("Saved:", configs_path.relative_to(PROJECT_ROOT))
print("selection_mode:", selection_mode)

Saved: reports\experiments_table.csv
Saved: reports\experiments_configs.json
selection_mode: sklearn_only_preliminary


In [15]:
# CÉLULA 15 — Selection Decision + gate final

decision_path = REPORTS_DIR / "selection_decision.md"
valid = experiments_df.dropna(subset=["val_f1_macro"]).copy()

selection_mode = infer_selection_mode_from_registry(EXPERIMENTS)

def fmt_md_value(v):
    if v is None:
        return "null"
    try:
        if pd.isna(v):
            return "null"
    except Exception:
        pass
    return v

lines = []
lines.append("# Selection Decision — Notebook 04 (Semana 4)\n")
lines.append("## Contexto\n")
lines.append("- Objetivo: executar experimentos rápidos, comparáveis e rastreáveis para orientar o Notebook 05.\n")
lines.append(f"- Contrato do target: `{target_cfg_source}`\n")
lines.append(f"- Seed global: `{GLOBAL_SEED}`\n")
lines.append(f"- default_device_available: `{DEVICE}`\n")
lines.append(f"- selection_mode: `{selection_mode}`\n")
if mask_policy is not None:
    lines.append(f"- mask_policy (contrato): `{mask_policy}`\n")

lines.append("\n## Critério de seleção\n")
lines.append("- Primário: **F1 macro em validação**.\n")
lines.append("- Secundário: accuracy, simplicidade operacional, estabilidade e tempo.\n")

lines.append("\n## Escopo desta decisão\n")
lines.append("**Trilha A: somente classificação.**\n")
lines.append("- Este Notebook 04 não executa segmentação; a etapa foi mantida fora para preservar comparabilidade e velocidade.\n")
if selection_mode == "sklearn_only_preliminary":
    lines.append("- O registry atual contém apenas experimentos **sklearn/sanity**.\n")
    lines.append("- Portanto, a seleção deste notebook é **preliminar** e serve como **baseline de referência para o Notebook 05**.\n")
    lines.append("- O resultado escolhido aqui **não deve ser tratado como arquitetura final oficial do projeto**.\n")
else:
    lines.append("- O registry atual inclui candidatos além de sanity baselines; a seleção pode ser usada como candidata ao treino final.\n")

if len(valid) == 0:
    lines.append("\n## Resultado\n")
    lines.append("**Nenhum experimento produziu métricas válidas.**\n")
    if "error" in experiments_df.columns:
        lines.append("\n### Erros observados (resumo)\n")
        vc = experiments_df["error"].dropna().value_counts().head(10)
        if len(vc) == 0:
            lines.append("- Nenhum erro resumido disponível.\n")
        else:
            for k, v in vc.items():
                lines.append(f"- {k}: {v}\n")

    lines.append("\n## Próximos passos\n")
    lines.append("- Corrigir a causa raiz e reexecutar o Notebook 04.\n")
    gate_pass = False
else:
    best = valid.iloc[0].to_dict()
    best_id = best["experiment_id"]
    best_cfg = configs_out.get(best_id, {})

    best_scope = str(best_cfg.get("selection_scope", best.get("selection_scope", "sanity_reference")))
    selected_role = (
        "baseline_reference_for_nb05"
        if best_scope == "sanity_reference"
        else "candidate_for_nb05"
    )

    lines.append("\n## Resultado (melhor experimento)\n")
    lines.append(f"- **experiment_id**: `{best_id}`\n")
    lines.append(f"- family: `{best.get('family')}`\n")
    lines.append(f"- selection_scope: `{best_scope}`\n")
    lines.append(f"- execution_device: `{best.get('device')}`\n")
    lines.append(f"- input_size: `{best.get('input_size')}`\n")
    lines.append(f"- class_weight: `{fmt_md_value(best.get('class_weight'))}`\n")
    lines.append(f"- val_f1_macro: `{best.get('val_f1_macro')}`\n")
    lines.append(f"- val_accuracy: `{best.get('val_accuracy')}`\n")
    lines.append(f"- runtime_sec: `{best.get('runtime_sec')}`\n")
    if best.get("val_confusion_matrix_path"):
        lines.append(f"- confusion_matrix plot: `{best.get('val_confusion_matrix_path')}`\n")

    lines.append("\n## Papel da config escolhida\n")
    lines.append(f"- selected_role: `{selected_role}`\n")
    if selected_role == "baseline_reference_for_nb05":
        lines.append("- Esta config deve entrar no Notebook 05 como **baseline de referência**, não como decisão definitiva de arquitetura CNN.\n")
    else:
        lines.append("- Esta config pode ser promovida para treino final no Notebook 05, desde que a execução permaneça reproduzível.\n")

    lines.append("\n## Config escolhida\n```json\n")
    lines.append(json.dumps(best_cfg, indent=2, ensure_ascii=False))
    lines.append("\n```\n")

    lines.append("\n## Interpretação dos resultados\n")
    if selected_role == "baseline_reference_for_nb05":
        lines.append("- O macro-F1 observado deve ser lido como **sanity check** de pipeline e baseline de comparação.\n")
        lines.append("- Antes de promover uma arquitetura final, a trilha de CNN deve ser comparada de forma limpa e rastreável.\n")
    else:
        lines.append("- A escolha prioriza F1 macro e mantém comparabilidade entre candidatos executados neste ambiente.\n")

    lines.append("\n## Próximos passos (Notebook 05)\n")
    lines.append(f"- Carregar `reports/experiments_configs.json` e registrar explicitamente o uso do experimento `{best_id}`.\n")
    if selected_role == "baseline_reference_for_nb05":
        lines.append("- Tratar esta config como baseline de referência; a promoção para modelo final depende de uma trilha de CNN limpa e reproduzível.\n")
    else:
        lines.append("- Treinar com mais rigor, salvar checkpoint do melhor modelo e exportar o pacote mínimo de inferência.\n")

    gate_pass = bool(table_path.exists() and configs_path.exists() and best_id is not None)

lines.append("\n## Gate final\n")
lines.append(f"- status: `{'PASS' if gate_pass else 'FAIL'}`\n")
lines.append(f"- experiments_table.csv salvo: `{table_path.exists()}`\n")
lines.append(f"- experiments_configs.json salvo: `{configs_path.exists()}`\n")
lines.append(f"- selection_decision.md salvo: `True`\n")

with open(decision_path, "w", encoding="utf-8") as f:
    f.write("".join(lines))

print("Saved:", decision_path.relative_to(PROJECT_ROOT))
print("GATE:", "PASS" if gate_pass else "FAIL")
if len(valid) > 0:
    print("Selected best experiment:", valid.iloc[0]["experiment_id"])

Saved: reports\selection_decision.md
GATE: PASS
Selected best experiment: sk_logreg_32
